# 03 — Feature Engineering

This notebook builds the modeling-ready dataset by:
1. Merging hourly pickups with weather and station metadata
2. Adding **temporal features** (hour, day-of-week, month, holidays, rush hour)
3. Adding **station features** (capacity, historical average demand)
4. Adding **weather-derived features** (is_rainy)
5. Adding **lag features** (1h, 24h, 1-week lookback)
6. **Chronological train/test split** (Jan–Aug train, Sep–Oct test)

Output: `data/processed/model_ready.csv`, `train.csv`, `test.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("..").resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Load processed data from notebook 02
hourly = pd.read_csv(PROCESSED_DIR / "hourly_pickups.csv", parse_dates=["hour"])
weather = pd.read_csv(PROCESSED_DIR / "weather_clean.csv", parse_dates=["hour"])
stations = pd.read_csv(PROCESSED_DIR / "stations_clean.csv")

print(f"Hourly pickups: {len(hourly):,} rows  ({hourly['station_id'].nunique()} stations × {hourly['hour'].nunique():,} hours)")
print(f"Weather: {len(weather):,} rows")
print(f"Stations: {len(stations)} rows")

Hourly pickups: 7,164,672 rows  (982 stations × 7,296 hours)
Weather: 7,296 rows
Stations: 1016 rows


## 1. Merge Weather Data
Join weather on the `hour` column — every station in the same hour gets the same weather (city-level data).

In [2]:
df = hourly.merge(weather, on="hour", how="left")

print(f"After weather merge: {len(df):,} rows")
print(f"Null check:\n{df.isnull().sum()}")
df.head()

After weather merge: 7,164,672 rows
Null check:
station_id          0
hour                0
pickups             0
temperature_c       0
precipitation_mm    0
wind_speed_kmh      0
weather_code        0
dtype: int64


,station_id,hour,pickups,temperature_c,precipitation_mm,wind_speed_kmh,weather_code
0,7000,2025-01-01 00:00:00,0,3.7,0.0,14.9,3
1,7000,2025-01-01 01:00:00,0,3.3,0.0,14.0,3
2,7000,2025-01-01 02:00:00,1,2.8,0.0,15.2,3
3,7000,2025-01-01 03:00:00,1,2.2,0.0,17.9,3
4,7000,2025-01-01 04:00:00,0,2.0,0.0,19.6,3


## 2. Merge Station Features
Add station capacity and lat/lon from metadata. Also compute historical average demand per station as a popularity proxy.

In [3]:
# Merge station metadata (capacity, lat, lon)
df = df.merge(
    stations[["station_id", "capacity", "lat", "lon"]],
    on="station_id",
    how="left"
)

# Compute station historical average demand (mean pickups across all hours)
station_avg = df.groupby("station_id")["pickups"].mean().rename("station_avg_pickups")
df = df.merge(station_avg, on="station_id", how="left")

print(f"After station merge: {len(df):,} rows")
print(f"Capacity nulls: {df['capacity'].isnull().sum()}")
print(f"\nStation avg pickups range: {df['station_avg_pickups'].min():.3f} – {df['station_avg_pickups'].max():.3f}")
df.head()

After station merge: 7,164,672 rows
Capacity nulls: 0

Station avg pickups range: 0.000 – 6.938


,station_id,hour,pickups,temperature_c,precipitation_mm,wind_speed_kmh,weather_code,capacity,lat,lon,station_avg_pickups
0,7000,2025-01-01 00:00:00,0,3.7,0.0,14.9,3,47,43.639832,-79.395954,4.346354
1,7000,2025-01-01 01:00:00,0,3.3,0.0,14.0,3,47,43.639832,-79.395954,4.346354
2,7000,2025-01-01 02:00:00,1,2.8,0.0,15.2,3,47,43.639832,-79.395954,4.346354
3,7000,2025-01-01 03:00:00,1,2.2,0.0,17.9,3,47,43.639832,-79.395954,4.346354
4,7000,2025-01-01 04:00:00,0,2.0,0.0,19.6,3,47,43.639832,-79.395954,4.346354


## 3. Temporal Features
Extract time-based features from the `hour` column. These capture daily, weekly, and seasonal demand patterns.

In [4]:
df["hour_of_day"] = df["hour"].dt.hour
df["day_of_week"] = df["hour"].dt.dayofweek       # 0=Mon, 6=Sun
df["month"] = df["hour"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Rush hour: weekday 7-9 AM or 4-7 PM
df["is_rush_hour"] = (
    (df["is_weekend"] == 0) &
    (
        df["hour_of_day"].between(7, 9) |
        df["hour_of_day"].between(16, 18)
    )
).astype(int)

# Ontario statutory holidays in 2025
ontario_holidays_2025 = pd.to_datetime([
    "2025-01-01",  # New Year's Day
    "2025-02-17",  # Family Day
    "2025-04-18",  # Good Friday
    "2025-05-19",  # Victoria Day
    "2025-07-01",  # Canada Day
    "2025-08-04",  # Civic Holiday
    "2025-09-01",  # Labour Day
    "2025-10-13",  # Thanksgiving
])
df["is_holiday"] = df["hour"].dt.normalize().isin(ontario_holidays_2025).astype(int)

print("Temporal features added:")
print(f"  hour_of_day: {df['hour_of_day'].min()}–{df['hour_of_day'].max()}")
print(f"  day_of_week: {df['day_of_week'].min()}–{df['day_of_week'].max()}")
print(f"  month: {df['month'].min()}–{df['month'].max()}")
print(f"  is_weekend: {df['is_weekend'].mean():.1%} of rows")
print(f"  is_rush_hour: {df['is_rush_hour'].mean():.1%} of rows")
print(f"  is_holiday: {df['is_holiday'].sum():,} rows ({df['is_holiday'].mean():.2%})")

Temporal features added:
  hour_of_day: 0–23
  day_of_week: 0–6
  month: 1–10
  is_weekend: 28.3% of rows
  is_rush_hour: 17.9% of rows
  is_holiday: 188,544 rows (2.63%)


## 4. Weather-Derived Features
Add a binary `is_rainy` flag from precipitation data.

In [5]:
df["is_rainy"] = (df["precipitation_mm"] > 0).astype(int)

print(f"Rainy hours: {df['is_rainy'].mean():.1%} of rows")
print(f"Precipitation when rainy — mean: {df.loc[df['is_rainy'] == 1, 'precipitation_mm'].mean():.2f} mm")

Rainy hours: 13.6% of rows
Precipitation when rainy — mean: 0.70 mm


## 5. Lag Features
Add demand from the same station at previous time steps. These capture autocorrelation in demand patterns.

- `lag_1h`: pickups at the same station 1 hour ago
- `lag_24h`: pickups at the same station 24 hours ago (same hour yesterday)
- `lag_168h`: pickups at the same station 168 hours ago (same hour last week)

**Important**: We sort by station and time, then use `groupby().shift()` to avoid data leakage across stations.

In [6]:
# Sort by station then time for correct shifting
df = df.sort_values(["station_id", "hour"]).reset_index(drop=True)

# Create lag features within each station
grouped = df.groupby("station_id")["pickups"]
df["lag_1h"] = grouped.shift(1)
df["lag_24h"] = grouped.shift(24)
df["lag_168h"] = grouped.shift(168)  # 7 days × 24 hours

print("Lag feature nulls (expected for first hours of each station):")
print(f"  lag_1h:   {df['lag_1h'].isnull().sum():,}")
print(f"  lag_24h:  {df['lag_24h'].isnull().sum():,}")
print(f"  lag_168h: {df['lag_168h'].isnull().sum():,}")

# Fill lag nulls with 0 (start-of-series, no prior data available)
df["lag_1h"] = df["lag_1h"].fillna(0)
df["lag_24h"] = df["lag_24h"].fillna(0)
df["lag_168h"] = df["lag_168h"].fillna(0)

print(f"\nLag features filled. Nulls remaining: {df[['lag_1h', 'lag_24h', 'lag_168h']].isnull().sum().sum()}")

Lag feature nulls (expected for first hours of each station):
  lag_1h:   982
  lag_24h:  23,568
  lag_168h: 164,976

Lag features filled. Nulls remaining: 0


## 6. Final Review
Check the full feature set, dtypes, and null counts before saving.

In [7]:
print(f"Final dataset: {len(df):,} rows × {len(df.columns)} columns\n")
print("Columns and dtypes:")
print(df.dtypes.to_string())
print(f"\nNull counts:\n{df.isnull().sum().to_string()}")
print(f"\nSample rows:")
df.sample(5, random_state=42)

Final dataset: 7,164,672 rows × 21 columns

Columns and dtypes:
station_id                      int64
hour                   datetime64[ns]
pickups                         int64
temperature_c                 float64
precipitation_mm              float64
wind_speed_kmh                float64
weather_code                    int64
capacity                        int64
lat                           float64
lon                           float64
station_avg_pickups           float64
hour_of_day                     int32
day_of_week                     int32
month                           int32
is_weekend                      int64
is_rush_hour                    int64
is_holiday                      int64
is_rainy                        int64
lag_1h                        float64
lag_24h                       float64
lag_168h                      float64

Null counts:
station_id             0
hour                   0
pickups                0
temperature_c          0
precipitation_mm       0

,station_id,hour,pickups,temperature_c,precipitation_mm,wind_speed_kmh,weather_code,capacity,lat,lon,...,hour_of_day,day_of_week,month,is_weekend,is_rush_hour,is_holiday,is_rainy,lag_1h,lag_24h,lag_168h
1292214,7210,2025-02-04 06:00:00,0,-4.8,0.0,15.2,3,15,43.647500,-79.433056,...,6,1,2,0,0,0,0,0.0,0.0,0.0
7132650,8172,2025-07-05 18:00:00,0,30.2,0.0,14.2,1,23,43.674857,-79.460192,...,18,5,7,1,0,0,0,0.0,0.0,0.0
2184867,7362,2025-05-21 03:00:00,0,8.6,0.0,15.0,3,19,43.683296,-79.418734,...,3,2,5,0,0,0,0,0.0,0.0,0.0
686937,7111,2025-02-16 09:00:00,0,-6.0,1.1,21.6,73,11,43.640885,-79.416379,...,9,6,2,1,0,0,1,0.0,0.0,0.0
5100737,7853,2025-02-04 17:00:00,0,-7.6,0.0,18.6,2,15,43.695498,-79.488094,...,17,1,2,0,1,0,0,0.0,0.0,0.0


## 7. Chronological Train/Test Split
- **Train**: Jan–Aug 2025 (~80%) — model sees both winter and summer patterns
- **Test**: Sep–Oct 2025 (~20%) — fall data, unseen during training

This prevents data leakage from future → past.

In [8]:
split_date = pd.Timestamp("2025-09-01")

train = df[df["hour"] < split_date].copy()
test = df[df["hour"] >= split_date].copy()

print(f"Train: {len(train):,} rows ({len(train)/len(df):.1%})")
print(f"  Date range: {train['hour'].min()} → {train['hour'].max()}")
print(f"Test:  {len(test):,} rows ({len(test)/len(df):.1%})")
print(f"  Date range: {test['hour'].min()} → {test['hour'].max()}")

# Sanity check: no overlap
assert train["hour"].max() < test["hour"].min(), "Train/test overlap detected!"
print("\nNo overlap — split is clean.")

Train: 5,727,024 rows (79.9%)
  Date range: 2025-01-01 00:00:00 → 2025-08-31 23:00:00
Test:  1,437,648 rows (20.1%)
  Date range: 2025-09-01 00:00:00 → 2025-10-31 23:00:00

No overlap — split is clean.


## 8. Define Feature Groups
Define the feature columns for each ablation level. These will be used in the modeling notebooks.

In [9]:
STATION_FEATURES = ["capacity", "station_avg_pickups"]

TEMPORAL_FEATURES = ["hour_of_day", "day_of_week", "month", "is_weekend", "is_rush_hour", "is_holiday"]

WEATHER_FEATURES = ["temperature_c", "precipitation_mm", "wind_speed_kmh", "is_rainy"]

LAG_FEATURES = ["lag_1h", "lag_24h", "lag_168h"]

ALL_FEATURES = STATION_FEATURES + TEMPORAL_FEATURES + WEATHER_FEATURES + LAG_FEATURES

TARGET = "pickups"

print("Feature groups:")
print(f"  Station:  {STATION_FEATURES}")
print(f"  Temporal: {TEMPORAL_FEATURES}")
print(f"  Weather:  {WEATHER_FEATURES}")
print(f"  Lag:      {LAG_FEATURES}")
print(f"\n  Total features: {len(ALL_FEATURES)}")
print(f"  Target: {TARGET}")

Feature groups:
  Station:  ['capacity', 'station_avg_pickups']
  Temporal: ['hour_of_day', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour', 'is_holiday']
  Weather:  ['temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'is_rainy']
  Lag:      ['lag_1h', 'lag_24h', 'lag_168h']

  Total features: 15
  Target: pickups


## 9. Save Outputs

In [10]:
# Save full dataset + train/test splits
df.to_csv(PROCESSED_DIR / "model_ready.csv", index=False)
train.to_csv(PROCESSED_DIR / "train.csv", index=False)
test.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("Saved to data/processed/:")
for f in ["model_ready.csv", "train.csv", "test.csv"]:
    size_mb = (PROCESSED_DIR / f).stat().st_size / 1e6
    print(f"  {f} — {size_mb:.1f} MB")

Saved to data/processed/:
  model_ready.csv — 866.1 MB
  train.csv — 691.9 MB
  test.csv — 174.2 MB
